# Perform an inventory audit of an Excel HTA model

This is attempting to recreate the first stage of the Microsoft CoPilot prompt i.e. identifying all inputs. This includes inputs that are hardcoded into formulas.

The diagram below illustrates the general approach (subject to change):

<img src='images/model_qc_workflow.png' width='800' height='800' />





## 1. Imports

### 1.1. General imports

In [1]:
import sys
from pathlib import Path
from rich import print as printr
import json
import pandas as pd

import warnings
# to ignore a openpyxl UserWarning
warnings.filterwarnings('ignore')  # Suppress all warnings

### 1.2 `xl-ray`

In [2]:
from xl_ray.extractor import (
    load_workbooks, 
    extract_metadata, 
    extract_vba_modules, 
    extract_named_ranges,
    extract_worksheets 
)

from xl_ray.schema import ExcelModelData

from xl_ray.audit import (
    detect_magic_numbers, # TM note - I wonder if this is useful for LLM hint? 
)

### 1.3 LLM workflow imports

In [3]:
# standard langchain 
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# custom in input_selection module
from input_selection import (
    SELECTION_INSTRUCTIONS,
    SheetSelection,
    build_selection_payload,
    selected_sheet_names,
    validate_selection,
    build_selection_text
)

## 2. Import model via `xl-ray`

In [4]:
TEST_MODEL = "models/Exeter_model_template_Mar25.xlsm"

In [5]:
test_file = Path(TEST_MODEL)

if not test_file.exists():
    print(f"Error: Could not find {test_file}")
    sys.exit(1)

print(f"Loading '{test_file.name}'...")
wb_formulas, wb_values = load_workbooks(test_file)

print("Extracting VBA modules...")
vba_modules = extract_vba_modules(test_file, wb_formulas)

print("Extracting Named Ranges...")
named_ranges = extract_named_ranges(wb_formulas)

print("Extracting Worksheets, Cells, and Tables (this may take a moment)...")
worksheets = extract_worksheets(wb_formulas, wb_values)

metadata = extract_metadata(
    path=Path(test_file),
    wb_formulas=wb_formulas, 
    vba_modules=vba_modules, 
    named_ranges=named_ranges
)

metadata.has_data_tables = any(ws.tables for ws in worksheets.values())
metadata.has_array_formulas = any(
    cell.is_array_formula for ws in worksheets.values() for cell in ws.cells.values()
)

excel_model = ExcelModelData(
    metadata=metadata,
    worksheets=worksheets,
    named_ranges=named_ranges,
    vba_modules=vba_modules
)

print("Import of Excel model complete.")

Loading 'Exeter_model_template_Mar25.xlsm'...
Extracting VBA modules...
Extracting Named Ranges...
Extracting Worksheets, Cells, and Tables (this may take a moment)...
Import of Excel model complete.


## 3. LLM functions

In [6]:
def select_worksheets_using_LLM(
    payload: str,
    system_prompt: str,
    base_url: str = "http://127.0.0.1:8001/v1",
    selected_model: str = "google/gemma-4-26B-A4B-it",
    random_seed: int | None = None,
    temperature: float = 1.0,
) -> SheetSelection:
    """
    Select worksheets for detailed input identification.

    `prompt` should contain the compact workbook summary.

    Returns a validated SheetSelection, not an AIMessage.
    """
    model = ChatOpenAI(
        base_url=base_url,
        api_key="EMPTY",
        model=selected_model,
        temperature=temperature,
        **({"seed": random_seed} if random_seed is not None else {}),
    )

    # used to get structured output
    structured_model = model.with_structured_output(
        SheetSelection,
        method="json_schema",
        include_raw=False,
    )

    selection = structured_model.invoke(
        [
            SystemMessage(content=system_prompt.strip()),
            HumanMessage(content=payload),
        ]
    )

    return selection

## 4. Script to select sheets to include in audit

### 4.1. Build the worksheet selection prompt

In [7]:
print("Building prompt")
payload = build_selection_payload(
    excel_model,
    max_rows_per_sheet = 12,
    max_cells_per_row = 10,
)

# convert JSON to plain text to reduce charactor count
workbook_summary_as_text= build_selection_text(payload)

print("Workbook summary characters:", len(workbook_summary_as_text))

Building prompt
Workbook summary characters: 71907


### 4.2 Prompt LLM to select worksheets

In [ ]:
%%time

print("LLM is selecting sheets to include in audit")
selection = select_worksheets_using_LLM(
    payload=workbook_summary_as_text,
    system_prompt = SELECTION_INSTRUCTIONS,
    random_seed=42,
)

LLM is selecting sheets to include in audit


### 4.3. Save results and convert to `DataFrame` 

In [ ]:
# Validate worksheet names and coverage.
selection = validate_selection(
    selection.model_dump_json(),
    excel_model,
)

sheets_to_audit = selected_sheet_names(
    selection,
    include_review=True,
)

# Convert the validated Pydantic objects to a DataFrame.
selection_df = pd.DataFrame(
    [
        {
            "sheet": item.sheet,
            "decision": item.decision,
            "reason": item.reason,
            "evidence": "\n".join(item.evidence),
            "uncertainty": item.uncertainty,
        }
        for item in selection.sheets
    ],
    columns=[
        "sheet",
        "decision",
        "reason",
        "evidence",
        "uncertainty",
    ],
)
 
selection_df["selected_for_audit"] = selection_df["sheet"].isin(
    sheets_to_audit
)

# Retain the worksheet objects for the next LLM stage.
selected_worksheets = {
    name: excel_model.worksheets[name]
    for name in sheets_to_audit
}

In [ ]:
# show sheets selected for audit
selection_df[selection_df["selected_for_audit"] == True]

In [ ]:
copilot_included = [
    "Model_settings",
    "Treatment_costs",
    "HCRU",
    "Societal_costs",
    "Effectiveness",
    "Adverse_events",
    "HRQoL",
    "LifeTable",
    "Parameters",
    "PSA",
    "OWSA",
    "Scenario_analysis",
    "Engine(1)",
    "Engine(2)",
    "Engine(3)",
]

len(copilot_included)

## End of Selection of Sheets.  